In [1]:
# --- project bootstrap (added during repo consolidation) ---
# Resolves the project root so the notebook runs from notebooks/ or from the repo root,
# and puts src/ on sys.path so `import laptop_price` works.
import os, sys, pathlib

_here = pathlib.Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError("pyproject.toml not found - run this notebook inside the repo")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"project root: {PROJECT_ROOT}")


project root: c:\Users\HP\Desktop\LAPTOP PRICE PREDICTION\laptop-price-prediction


# Clustering Models

## Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, learning_curve
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

c:\Users\HP\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
df = pd.read_csv("data/processed/model_ready_data.csv", index_col=0)

# Handle outliers using IQR method on target
Q1 = df['price_preview'].quantile(0.01)
Q3 = df['price_preview'].quantile(0.99)
df_clean = df[(df['price_preview'] >= Q1) & (df['price_preview'] <= Q3)]



X = df_clean.drop('price_preview', axis=1)
y = np.log1p(df_clean['price_preview'])  # Log transform for large values

# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Use RobustScaler for better handling of outliers
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Original samples: {len(df)}, After outlier removal: {len(df_clean)}")
print(f"Train: {X_train.shape[0]}, Validation: {X_val.shape[0]}, Test: {X_test.shape[0]}")

Original samples: 15517, After outlier removal: 15209
Train: 10646, Validation: 2281, Test: 2282


## Clustering Models

In [4]:
df.isna().sum()

price_preview            0
RAM_SIZE                 0
SSD_SIZE                 0
HDD_SIZE                 0
cores                    0
cpu_mark                 0
tdp                      0
gpu_g3d_mark             0
gpu_g2d_mark             0
gpu_tdp                  0
SCREEN_SIZE_SNAPPED      0
SCREEN_RESOLUTION_ENC    0
model_family             0
RAM_TYPE                 0
spec_Etat                0
dtype: int64

In [5]:
# Spec-based clustering: exclude price to avoid circular reasoning
X_clust_source = X.copy()
# Note: price_preview is intentionally excluded from clustering features.
# Including price would make the segments price-defined rather than spec-defined.
cluster_scaler = RobustScaler()
X_all_scaled = cluster_scaler.fit_transform(X_clust_source)

# Find optimal number of clusters using Elbow method and Silhouette scores
silhouette_scores = []
inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans_temp.fit(X_all_scaled)
    silhouette_scores.append(silhouette_score(X_all_scaled, kmeans_temp.labels_, sample_size=5000, random_state=42))
    inertias.append(kmeans_temp.inertia_)

# Find best k based on silhouette score
best_k = k_range[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters (by silhouette): {best_k}")
print(f"Silhouette scores per k: {dict(zip(k_range, [f'{s:.4f}' for s in silhouette_scores]))}")

# 1. K-Means++ with optimal k
kmeans = KMeans(n_clusters=best_k, init='k-means++', n_init=20, max_iter=500, random_state=42)
kmeans_labels = kmeans.fit_predict(X_all_scaled)
kmeans_sil = silhouette_score(X_all_scaled, kmeans_labels, sample_size=5000, random_state=42)
print(f"\nK-Means++ (k={best_k}) Silhouette: {kmeans_sil:.4f}")

# 2. DBSCAN with parameter search
best_dbscan_sil = -1
best_eps, best_min_samples = 2.5, 5
for eps in [1.5, 2.0, 2.5, 3.0, 3.5]:
    for min_samples in [3, 5, 7, 10]:
        dbscan_temp = DBSCAN(eps=eps, min_samples=min_samples)
        labels_temp = dbscan_temp.fit_predict(X_all_scaled)
        n_clusters_temp = len(set(labels_temp)) - (1 if -1 in labels_temp else 0)
        if n_clusters_temp > 1:
            mask_temp = labels_temp != -1
            if sum(mask_temp) > 10:
                sil_temp = silhouette_score(X_all_scaled[mask_temp], labels_temp[mask_temp], sample_size=min(5000, mask_temp.sum()), random_state=42)
                if sil_temp > best_dbscan_sil:
                    best_dbscan_sil = sil_temp
                    best_eps, best_min_samples = eps, min_samples

dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples)
dbscan_labels = dbscan.fit_predict(X_all_scaled)
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
if n_clusters_dbscan > 1:
    mask = dbscan_labels != -1
    dbscan_sil = silhouette_score(X_all_scaled[mask], dbscan_labels[mask], sample_size=min(5000, mask.sum()), random_state=42)
    print(f"DBSCAN (eps={best_eps}, min_samples={best_min_samples}) Silhouette: {dbscan_sil:.4f}")
    print(f"  Clusters: {n_clusters_dbscan}, Noise points: {sum(dbscan_labels==-1)}")
else:
    dbscan_sil = 0
    print(f"DBSCAN found {n_clusters_dbscan} cluster(s) - no valid clustering")

# 3. Agglomerative Hierarchical with different linkages
best_agg_sil = -1
best_linkage = 'ward'
for linkage in ['ward', 'complete', 'average']:
    agg_temp = AgglomerativeClustering(n_clusters=best_k, linkage=linkage)
    labels_temp = agg_temp.fit_predict(X_all_scaled)
    sil_temp = silhouette_score(X_all_scaled, labels_temp, sample_size=5000, random_state=42)
    if sil_temp > best_agg_sil:
        best_agg_sil = sil_temp
        best_linkage = linkage

agg = AgglomerativeClustering(n_clusters=best_k, linkage=best_linkage)
agg_labels = agg.fit_predict(X_all_scaled)
agg_sil = silhouette_score(X_all_scaled, agg_labels, sample_size=5000, random_state=42)
print(f"Agglomerative ({best_linkage}) Silhouette: {agg_sil:.4f}")

In [ ]:
# Summarize and visualize K-Means with k=3 and k=4
feature_variance = X_clust_source.var().sort_values(ascending=False)
top_features = feature_variance.head(8).index.tolist()
summary_cols = ["price_preview"] + [c for c in top_features if c != "price_preview"]

from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_all_scaled)

for k in [3, 4]:
    kmeans_k = KMeans(n_clusters=k, init="k-means++", n_init=20, max_iter=500, random_state=42)
    labels_k = kmeans_k.fit_predict(X_all_scaled)

    cluster_df_k = X_clust_source.copy()
    cluster_df_k["cluster"] = labels_k

    cluster_sizes_k = cluster_df_k["cluster"].value_counts().sort_index()
    print(f"K-Means (k={k}) cluster sizes:")
    display(cluster_sizes_k.to_frame(name="count"))

    cluster_summary_k = cluster_df_k.groupby("cluster")[summary_cols].agg(["mean", "median", "min", "max"])
    display(cluster_summary_k)

    plt.figure(figsize=(8, 4))
    sns.barplot(x=cluster_sizes_k.index, y=cluster_sizes_k.values, color="#4C72B0")
    plt.title(f"K-Means Cluster Sizes (k={k})")
    plt.xlabel("Cluster")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    sns.boxplot(data=cluster_df_k, x="cluster", y="price_preview")
    plt.title(f"Price Distribution by Cluster (k={k})")
    plt.xlabel("Cluster")
    plt.ylabel("Price (preview)")
    plt.tight_layout()
    plt.show()

    pca_df_k = pd.DataFrame(pca_coords, columns=["PC1", "PC2"])
    pca_df_k["cluster"] = labels_k

    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=pca_df_k, x="PC1", y="PC2", hue="cluster", palette="tab10", s=40, alpha=0.8)
    plt.title(f"K-Means Clusters (PCA 2D, k={k})")
    plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

> **Warning — Degenerate Clustering**: K-Means with k=3 on this dataset produces a highly unbalanced result (~93% of points in one cluster). This is a known issue: the spec distribution is not well-suited to K-Means on raw features.

> **See notebook 10** (`10_market_segmentation.ipynb`) for the production-quality consensus segmentation that produces stable, meaningful market segments using Gower distance and HDBSCAN.

## Model Summary & Presentation Charts

The following section consolidates the main model results and provides presentation-ready visuals (performance comparison, residuals, and learning curves).

In [ ]:
# NOTE: Cells referencing regression variables (baseline, be_r2, tuned_r2)
# from notebook 04 have been removed. This notebook focuses on clustering only.
# For regression comparisons, see notebook 04_regression.ipynb or notebook 12_model_comparison.ipynb.
print('Clustering analysis complete. See notebook 10 for production segmentation.')